In [1]:
!pip install --quiet qiskit
!pip install --quiet mqt.bench

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 123.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 142.3 kB/s eta 0:00:00


In [2]:
!wget https://packages.microsoft.com/config/ubuntu/20.04/packages-microsoft-prod.deb -O packages-microsoft-prod.deb
!dpkg -i packages-microsoft-prod.deb
!apt-get update
!apt-get install -y dotnet-sdk-8.0

!pip install --quiet qsharp azure-quantum

--2026-04-06 08:12:33--  https://packages.microsoft.com/config/ubuntu/20.04/packages-microsoft-prod.deb
Resolving packages.microsoft.com (packages.microsoft.com)... 13.107.213.73, 13.107.246.73, 2620:1ec:bdf::73, ...
Connecting to packages.microsoft.com (packages.microsoft.com)|13.107.213.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3690 (3.6K) [application/octet-stream]
Saving to: ‘packages-microsoft-prod.deb’

packages-microsoft- 100%[===================>]   3.60K  --.-KB/s    in 0s      

2026-04-06 08:12:33 (1.35 GB/s) - ‘packages-microsoft-prod.deb’ saved [3690/3690]

Selecting previously unselected package packages-microsoft-prod.
(Reading database ... 120314 files and directories currently installed.)
Preparing to unpack packages-microsoft-prod.deb ...
Unpacking packages-microsoft-prod (1.0-ubuntu20.04.1) ...
Setting up packages-microsoft-prod (1.0-ubuntu20.04.1) ...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qsharp.estimator import EstimatorParams, ErrorBudgetPartition, LogicalCounts
from qsharp.interop.qiskit import estimate
from mqt.bench import get_benchmark, BenchmarkLevel

QISKIT_STD_GATES = [
    "p", "x", "y", "z", "h", "s", "sdg", "t", "tdg", "sx",
    "rx", "ry", "rz", "cx", "cy", "cz", "cp", "crx", "cry",
    "crz", "ch", "ccx", "cu", "id", "u1", "u2", "u3"
]


def estimate_resources(logical_counts, error_budget_dist):
    """Estimates physical resources for a given error budget distribution."""
    params = EstimatorParams()
    params.error_budget = ErrorBudgetPartition()
    params.error_budget.logical = error_budget_dist['logical']
    params.error_budget.t_states = error_budget_dist['t_states']
    params.error_budget.rotations = error_budget_dist['rotations']

    result = logical_counts.estimate(params=params)

    physical_qubits = result["physicalCounts"]["physicalQubits"]
    runtime = result["physicalCounts"]["runtime"]

    return {
        'physical_qubits': physical_qubits,
        'runtime': runtime,
        'metric': runtime * physical_qubits
    }


def compute_cost(qubits, runtime, weight_qubits=0.5):
    """Compute cost with proper normalization."""
    q = max(qubits, 1)
    r = max(runtime, 1)
    cost = (q ** weight_qubits) * (r ** (1 - weight_qubits))
    return cost





def simplex_to_y(p):
    """Simplex -> unconstrained R^2 via log-ratio."""
    p = np.maximum(p, 1e-15)
    return np.array([np.log(p[1] / p[0]), np.log(p[2] / p[0])])


def y_to_simplex(y):
    """Unconstrained R^2 -> simplex via softmax."""
    z = np.array([0.0, y[0], y[1]])
    z -= np.max(z)
    e = np.exp(z)
    return e / np.sum(e)


def apply_min_fraction(fracs, min_frac=0.05):
    """Ensure every component >= min_frac while preserving sum = 1.

    Reserves min_frac for each component, then distributes the
    remaining budget proportionally to the original fractions.
    This is a physical constraint: the Azure Resource Estimator
    requires code distance <= 50, which fails if any budget
    fraction is too small.
    """
    remainder = 1.0 - 3 * min_frac  # 0.85 with min_frac=0.05
    excess = np.maximum(fracs - min_frac, 0.0)
    s = np.sum(excess)
    if s > 0:
        return min_frac + excess / s * remainder
    else:
        return np.full(3, 1.0 / 3)


def qpso_optimization(logical_counts, total_budget,
                      pop_size=30, max_generations=150,
                      n_restarts=5,
                      weight_qubits=0.5,
                      beta_max=1.0, beta_min=0.5,
                      min_frac=0.05):
    """QPSO optimization of error budget distribution.

    Multi-restart QPSO operating in log-ratio space with proper
    simplex mapping. Faithful to Sun et al. (CEC 2004).

    Parameters
    ----------
    logical_counts : LogicalCounts
        Logical resource counts from the circuit.
    total_budget : float
        Total error budget.
    pop_size : int
        Particles per swarm.
    max_generations : int
        Generations per restart.
    n_restarts : int
        Independent QPSO runs (best result kept).
    weight_qubits : float
        Cost function exponent.
    beta_max, beta_min : float
        Contraction-expansion coefficient range.
    min_frac : float
        Minimum fraction per component (physical constraint).

    Returns
    -------
    dict with 'name', 'logical', 't_states', 'rotations'.
    """
    dim = 2  # 2-simplex has 2 DOF

    print(f"\n{'='*70}")
    print("QPSO Error Budget Optimization (Sun et al., IEEE CEC, 2004)")
    print(f"{'='*70}")
    print(f"Particles: {pop_size}, Generations: {max_generations}, "
          f"Restarts: {n_restarts}")
    print(f"Beta: {beta_max} -> {beta_min}, Min fraction: {min_frac}")
    print(f"Total evaluations budget: ~{pop_size * max_generations * n_restarts}")

    # ---- Evaluation helper ----
    eval_count = [0]

    def evaluate(y):
        """Map log-ratio point to simplex, evaluate resource estimator."""
        fracs = y_to_simplex(y)
        fracs = apply_min_fraction(fracs, min_frac)
        dist = {
            'logical':  fracs[0] * total_budget,
            't_states': fracs[1] * total_budget,
            'rotations': fracs[2] * total_budget
        }
        eval_count[0] += 1
        try:
            res = estimate_resources(logical_counts, dist)
            cost = compute_cost(res['physical_qubits'], res['runtime'],
                                weight_qubits)
            return cost, fracs, res['physical_qubits'], res['runtime']
        except Exception:
            return float('inf'), fracs, 0, 0

    # ---- Baseline ----
    bl_dist = {k: total_budget / 3 for k in ['logical', 't_states', 'rotations']}
    bl_res = estimate_resources(logical_counts, bl_dist)
    bl_cost = compute_cost(bl_res['physical_qubits'], bl_res['runtime'],
                           weight_qubits)
    print(f"Baseline (uniform): Q={bl_res['physical_qubits']:,}, "
          f"R={bl_res['runtime']:.3e}, Cost={bl_cost:.3e}")

    # ---- Global best across all restarts ----
    global_best_cost = float('inf')
    global_best_fracs = np.array([1/3, 1/3, 1/3])
    global_best_qubits = bl_res['physical_qubits']
    global_best_runtime = bl_res['runtime']

    rng = np.random.default_rng(210)

    for restart in range(n_restarts):

        # ---- Initialize particles ----
        # Dirichlet -> log-ratio gives diverse starting points
        simplex_init = rng.dirichlet(np.ones(3), size=pop_size)
        positions = np.array([simplex_to_y(s) for s in simplex_init])

        pbest_pos = positions.copy()
        pbest_cost = np.full(pop_size, float('inf'))

        gbest_pos = None
        gbest_cost = float('inf')
        gbest_fracs = None
        gbest_qubits = 0
        gbest_runtime = 0

        # Evaluate initial population
        for i in range(pop_size):
            cost, fracs, qb, rt = evaluate(positions[i])
            pbest_cost[i] = cost
            if cost < gbest_cost:
                gbest_cost = cost
                gbest_pos = positions[i].copy()
                gbest_fracs = fracs.copy()
                gbest_qubits = qb
                gbest_runtime = rt

        # ---- QPSO generations ----
        stagnation = 0
        prev_best = gbest_cost

        for gen in range(max_generations):
            # Beta linear decrease (Sun et al., 2004)
            beta = beta_max - (beta_max - beta_min) * gen / max(max_generations - 1, 1)

            # Mean best (Sun, Xu, Feng, 2004 — "Global Search Strategy")
            mbest = np.mean(pbest_pos, axis=0)

            for i in range(pop_size):
                # Local attractor (Eq. 3)
                phi = rng.uniform(0, 1, size=dim)
                p = phi * pbest_pos[i] + (1 - phi) * gbest_pos

                # Position update (Eq. 4) — the quantum tunneling step
                u = rng.uniform(1e-10, 1, size=dim)
                sign = np.where(rng.random(dim) < 0.5, 1.0, -1.0)
                new_pos = p + sign * beta * np.abs(mbest - positions[i]) * np.log(1.0 / u)

                # Evaluate
                cost, fracs, qb, rt = evaluate(new_pos)
                positions[i] = new_pos

                # Update personal best
                if cost < pbest_cost[i]:
                    pbest_cost[i] = cost
                    pbest_pos[i] = new_pos.copy()

                    # Update swarm best
                    if cost < gbest_cost:
                        gbest_cost = cost
                        gbest_pos = new_pos.copy()
                        gbest_fracs = fracs.copy()
                        gbest_qubits = qb
                        gbest_runtime = rt

            # Early stopping if stagnated
            if abs(gbest_cost - prev_best) < 1e-10:
                stagnation += 1
            else:
                stagnation = 0
            prev_best = gbest_cost

            if stagnation >= 30:
                print(f"  Restart {restart+1} converged at gen {gen+1}")
                break

        # Update global best across restarts
        if gbest_cost < global_best_cost:
            global_best_cost = gbest_cost
            global_best_fracs = gbest_fracs.copy()
            global_best_qubits = gbest_qubits
            global_best_runtime = gbest_runtime

        imp_r = ((bl_cost - gbest_cost) / bl_cost) * 100
        print(f"  Restart {restart+1}/{n_restarts}: cost={gbest_cost:.3e} "
              f"({imp_r:+.2f}%), fracs=[{gbest_fracs[0]:.4f}, "
              f"{gbest_fracs[1]:.4f}, {gbest_fracs[2]:.4f}]")

    # ---- Final result ----
    imp = ((bl_cost - global_best_cost) / bl_cost) * 100

    print(f"\n{'='*70}")
    print("QPSO Complete")
    print(f"{'='*70}")
    print(f"Evaluations: {eval_count[0]}")
    print(f"Optimal: [{global_best_fracs[0]:.6f}, {global_best_fracs[1]:.6f}, "
          f"{global_best_fracs[2]:.6f}]")
    print(f"  Logical:   {global_best_fracs[0]*total_budget:.8f}")
    print(f"  T-states:  {global_best_fracs[1]*total_budget:.8f}")
    print(f"  Rotations: {global_best_fracs[2]*total_budget:.8f}")
    print(f"  Q={global_best_qubits:,}, R={global_best_runtime:.3e}")
    print(f"  Cost={global_best_cost:.3e}, Improvement: {imp:+.2f}%")

    return {
        'name': 'QPSO',
        'logical':   global_best_fracs[0] * total_budget,
        't_states':  global_best_fracs[1] * total_budget,
        'rotations': global_best_fracs[2] * total_budget
    }


def process_single_circuit(benchmark_name, circuit_size, total_error_budget=0.1,
                          verbose=False):
    """Process a single circuit and return results."""
    try:
        qc = get_benchmark(benchmark=benchmark_name,
                           level=BenchmarkLevel.ALG,
                           circuit_size=circuit_size)
        transpiled_qc = transpile(qc, basis_gates=QISKIT_STD_GATES,
                                  optimization_level=3)
        estimation_result = estimate(transpiled_qc)
        logical_counts_dict = estimation_result["logicalCounts"]
        logical_counts = LogicalCounts(logical_counts_dict)

        import sys
        from io import StringIO
        if not verbose:
            old_stdout = sys.stdout
            sys.stdout = StringIO()

        optimal_distribution = qpso_optimization(
            logical_counts, total_error_budget,
            pop_size=20, max_generations=80,
            n_restarts=3, weight_qubits=0.5
        )

        if not verbose:
            sys.stdout = old_stdout

        optimal_result = estimate_resources(logical_counts, optimal_distribution)

        uniform_value = total_error_budget / 3
        uniform_distribution = {
            'logical': uniform_value,
            't_states': uniform_value,
            'rotations': uniform_value
        }
        uniform_result = estimate_resources(logical_counts, uniform_distribution)

        qubit_imp = ((uniform_result['physical_qubits'] -
                      optimal_result['physical_qubits'])
                     / uniform_result['physical_qubits']) * 100
        runtime_imp = ((uniform_result['runtime'] - optimal_result['runtime'])
                       / uniform_result['runtime']) * 100
        metric_imp = ((uniform_result['metric'] - optimal_result['metric'])
                      / uniform_result['metric']) * 100

        return {
            'benchmark': benchmark_name,
            'circuit_size': circuit_size,
            'original_depth': qc.depth(),
            'transpiled_depth': transpiled_qc.depth(),
            'logical_qubits': logical_counts_dict['numQubits'],
            't_gates': logical_counts_dict['tCount'],
            'rotations': logical_counts_dict['rotationCount'],
            'rotation_depth': logical_counts_dict['rotationDepth'],
            'ccz_gates': logical_counts_dict['cczCount'],
            'uniform_physical_qubits': uniform_result['physical_qubits'],
            'uniform_runtime': uniform_result['runtime'],
            'uniform_metric': uniform_result['metric'],
            'optimal_logical_budget': optimal_distribution['logical'],
            'optimal_t_states_budget': optimal_distribution['t_states'],
            'optimal_rotations_budget': optimal_distribution['rotations'],
            'optimal_physical_qubits': optimal_result['physical_qubits'],
            'optimal_runtime': optimal_result['runtime'],
            'optimal_metric': optimal_result['metric'],
            'qubit_improvement_pct': qubit_imp,
            'runtime_improvement_pct': runtime_imp,
            'metric_improvement_pct': metric_imp,
            'status': 'success'
        }

    except Exception as e:
        return {
            'benchmark': benchmark_name,
            'circuit_size': circuit_size,
            'status': 'failed',
            'error': str(e)
        }


def main():
    import pandas as pd
    from datetime import datetime

    benchmarks_config = [
        ("ghz", [3, 4, 5, 6, 7, 8, 9, 10, 15, 16, 17, 18, 19, 20,21,22,23,24, 25, 30]),
        ("graphstate", [3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40]),
        ("wstate", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25, 30]),
        ("qft", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,25,30]),
        ("qftentangled", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]),
        ("qpeexact", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,20,25]),
        ("qpeinexact", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,20,25]),
        ("dj", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 25, 30]),
        ("bv", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20,21,22,23,24, 25, 30]),
        ("grover", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
        ("qaoa", [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
        ("ae", [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]),
        ("hhl", [5, 7, 9, 11, 13, 15,17]),
        ("qnn", [4, 5, 6, 7, 8, 9, 10,11,12,13,14,15]),
        ("cdkm_ripple_carry_adder", [ 4, 8,12,14,16,18,20,22,24,26]),
        ("full_adder",[4,8,12,14,16,18,20,22,24,26,28]),
        ("half_adder",[3,5,7,9,11,13,15,17,19,21,23,25,27]),
        ("hrs_cumulative_multiplier",[5,9,13,17,21,25]),
        ("randomcircuit",[2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
        ("vqe_su2",[2,3,4,5,6,7,8,9,10,15,20,25,30,35,40,80]),
        ("vqe_two_local",[2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
        ("qwalk",[4,5,6,7,8,9,10,11,12,13,14,15]),
        ("shor", [18,42]),
        ("modular_adder",[2,4,6,8,10,12,14,16,18,20,22,24,26,28,30]),
        ("vqe_real_amp",[2,3,4,5,6,7,8,9,10,11,12,13,14,15,35,40,45,50,55,60]),
        ("bmw_quark_cardinality",[2,3,4,5,10,11,12,13,14,15,40,60]),
        ("bmw_quark_copula",[2,4,6,8,10,12,14,16,18,20,28,30,50,60]),
        ("rg_qft_multiplier",[4,8,12,16,24,28,32]),
        ("draper_qft_adder",[2,4,6,8,10,14,16,18,20,22,24,26,28,30,40,50,60,90]),
        ("multiplier",[4,8,12,16,20,24,28,32,36,40,44,48,52,60]),
        ("vbe_ripple_carry_adder",[4,7,10,13,16,19,22,25,31,34,37,40,43,46,61,91]),
    ]

    print("="*70)
    print("MQT Benchmark Suite - Resource Estimation Analysis")
    print("="*70)
    print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    total_circuits = sum(len(sizes) for _, sizes in benchmarks_config)
    print(f"Benchmarks: {len(benchmarks_config)}, Circuits: {total_circuits}")
    print("="*70)

    all_results = []
    circuit_count = 0

    for benchmark_name, circuit_sizes in benchmarks_config:
        for circuit_size in circuit_sizes:
            circuit_count += 1
            print(f"\n[{circuit_count}/{total_circuits}] "
                  f"{benchmark_name} (size={circuit_size})...", end=" ")

            result = process_single_circuit(
                benchmark_name, circuit_size,
                total_error_budget=0.1, verbose=False
            )
            all_results.append(result)

            if result['status'] == 'success':
                print(f"✓ ({result['metric_improvement_pct']:+.2f}%)")
            else:
                print(f"✗ ({result.get('error', 'Unknown')})")

    df = pd.DataFrame(all_results)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_filename = f'mqt_benchmark_results_{timestamp}.csv'
    df.to_csv(csv_filename, index=False)
    print(f"\nResults saved to: {csv_filename}")

    successful = df[df['status'] == 'success']
    if len(successful) > 0:
        print(f"\n{'='*70}")
        print("RESULTS - QPSO vs UNIFORM DISTRIBUTION")
        print(f"{'='*70}")
        print(f"Success: {len(successful)}, Failed: {len(df)-len(successful)}")

        for label, col in [("Physical Qubits", "qubit_improvement_pct"),
                           ("Runtime", "runtime_improvement_pct"),
                           ("Overall Metric", "metric_improvement_pct")]:
            print(f"\n{label}:")
            print(f"  Mean:   {successful[col].mean():+.2f}%")
            print(f"  Median: {successful[col].median():+.2f}%")
            print(f"  Range:  [{successful[col].min():+.2f}%, "
                  f"{successful[col].max():+.2f}%]")

        best = successful.loc[successful['metric_improvement_pct'].idxmax()]
        worst = successful.loc[successful['metric_improvement_pct'].idxmin()]
        print(f"\nBest:  {best['benchmark']} (size={int(best['circuit_size'])})"
              f" — {best['metric_improvement_pct']:+.2f}%")
        print(f"Worst: {worst['benchmark']} (size={int(worst['circuit_size'])})"
              f" — {worst['metric_improvement_pct']:+.2f}%")

        avg = successful['metric_improvement_pct'].mean()
        if avg > 0:
            print(f"\n✓ QPSO achieves {avg:.2f}% average improvement over "
                  f"uniform across {len(successful)} circuits.")
        elif avg < 0:
            print(f"\n✗ QPSO shows {abs(avg):.2f}% degradation vs uniform.")
        else:
            print(f"\n≈ QPSO performs similarly to uniform.")

    print(f"\nEnd: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


if __name__ == "__main__":
    main()

MQT Benchmark Suite - Resource Estimation Analysis
Start time: 2026-04-06 08:14:10
Benchmarks: 31, Circuits: 433

[1/433] ghz (size=3)... ✓ (+0.00%)

[2/433] ghz (size=4)... ✓ (+0.00%)

[3/433] ghz (size=5)... ✓ (+0.00%)

[4/433] ghz (size=6)... ✓ (+78.40%)

[5/433] ghz (size=7)... ✓ (+78.40%)

[6/433] ghz (size=8)... ✓ (+78.40%)

[7/433] ghz (size=9)... ✓ (+78.40%)

[8/433] ghz (size=10)... ✓ (+78.40%)

[9/433] ghz (size=15)... ✓ (+0.00%)

[10/433] ghz (size=16)... ✓ (+0.00%)

[11/433] ghz (size=17)... ✓ (+0.00%)

[12/433] ghz (size=18)... ✓ (+0.00%)

[13/433] ghz (size=19)... ✓ (+0.00%)

[14/433] ghz (size=20)... ✓ (+0.00%)

[15/433] ghz (size=21)... ✓ (+63.56%)

[16/433] ghz (size=22)... ✓ (+63.56%)

[17/433] ghz (size=23)... ✓ (+63.56%)

[18/433] ghz (size=24)... ✓ (+63.56%)

[19/433] ghz (size=25)... ✓ (+63.56%)

[20/433] ghz (size=30)... ✓ (+63.56%)

[21/433] graphstate (size=3)... ✓ (+0.00%)

[22/433] graphstate (size=4)... ✓ (+0.00%)

[23/433] graphstate (size=5)... ✓ (+0.00%)
